In [1]:
# basic
import os
import pickle
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
# pre processing
from sklearn import preprocessing as pre
# NN
import torch
import torch.nn as nn
from torch import Tensor
import torch.nn.functional as F
import torch.optim as optim
from torch.nn import MSELoss
from torch_geometric.nn import GCNConv
# val and plot
#from torchmetrics.regression import R2Score
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error
from loguru import logger as log
# plot
import matplotlib.pyplot as plt
# foundation model
from functools import reduce

/home/marcos/.pyenv/versions/3.10.13/envs/gnn-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
plt.style.use("seaborn-v0_8-whitegrid")
SEED = 1345
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
seed_everything(SEED)
#plt.style.use('seaborn-whitegrid')
#pd.set_option('display.float_format', '{:.16f}'.format)
warnings.filterwarnings('ignore')

In [3]:
def load_datasets(filepath):
    """Carrega os datasets de arquivos pickle."""
    try:
        with open(filepath, 'rb') as f:
            dataset = pickle.load(f)
        return dataset
    except IOError as e:
        log.error(f"Erro ao carregar o dataset: {e}")
    except pickle.PickleError as e:
        log.error(f"Erro ao desserializar o dataset: {e}")
        traceback.print_exception(e)

In [4]:
sb = pd.read_parquet("/home/marcos/loader_03-04_2024.parquet")
sb.head()

,125960550,230565994,258781031,43768720,44072192,44783654,44783914,44784438,45833547,47568123
2024-03-01 05:00:00,1.333333,0.000000,70.792221,41.188599,1.724359,14.955100,2.032506,3.571820,5.877792,7.546274
2024-03-01 05:30:00,3.595238,1.583333,229.051071,172.071198,8.282966,44.559937,11.048912,18.211931,18.912033,18.276293
2024-03-01 06:00:00,4.812975,3.268518,424.853729,433.062469,18.825665,97.263435,26.276600,41.471294,40.731876,37.141144
2024-03-01 06:30:00,9.215629,5.256614,630.444153,743.177368,25.593414,149.329544,49.763138,71.520836,57.200085,53.487366
2024-03-01 07:00:00,12.585028,6.152447,841.874512,1132.739502,44.350349,204.275940,78.721497,107.241295,77.808769,75.446609


In [5]:
serie = sb["125960550"]

In [6]:
total_points = sb.shape[0]
t = np.arange(total_points)

In [7]:
limite_marco = 31 * 40
limite_abril = limite_marco + (30 * 40)
limite_marco, limite_abril

(1240, 2440)

In [8]:
future_steps = sb.shape[0] - limite_marco
future_steps

2400

In [9]:
sb["125960550"][limite_marco:].shape

(2400,)

In [10]:
input_size = 1       # entrada por passo de tempo
hidden_size = 32     # tamanho do hidden state da GRU
seq_len = 40 * 7         # número de passos anteriores usados para prever

train_points = limite_marco  # 1 mês
future_steps = future_steps  
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [11]:
def create_sequences(data, seq_len):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(serie[:train_points], seq_len)

X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1).to(device)  # (batch, seq_len, 1)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)   # (batch, 1)
X_train.shape, y_train.shape

(torch.Size([960, 280, 1]), torch.Size([960, 1]))

In [12]:
X_train.shape, y_train.shape

(torch.Size([960, 280, 1]), torch.Size([960, 1]))

In [13]:
X_train[0].shape

torch.Size([280, 1])

## Model

In [14]:
class GRUNet(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(GRUNet, self).__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, h = self.gru(x)  # h: (1, batch, hidden)
        out = self.fc(h.squeeze(0))  # (batch, hidden) -> (batch, 1)
        return out

In [15]:
model = GRUNet(input_size=1, hidden_size=hidden_size).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [16]:
# --- 5. Treinamento
epochs = 500
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    output = model(X_train)
    loss = criterion(output, y_train)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

Epoch 10/500, Loss: 197.5882
Epoch 20/500, Loss: 184.3878
Epoch 30/500, Loss: 171.0034
Epoch 40/500, Loss: 157.8257
Epoch 50/500, Loss: 145.2631
Epoch 60/500, Loss: 133.9726
Epoch 70/500, Loss: 124.5603
Epoch 80/500, Loss: 116.5165
Epoch 90/500, Loss: 109.3399
Epoch 100/500, Loss: 103.2275
Epoch 110/500, Loss: 97.8819
Epoch 120/500, Loss: 93.0337
Epoch 130/500, Loss: 88.5849
Epoch 140/500, Loss: 84.4788
Epoch 150/500, Loss: 80.6794
Epoch 160/500, Loss: 77.1599
Epoch 170/500, Loss: 73.8984
Epoch 180/500, Loss: 70.8757
Epoch 190/500, Loss: 68.0741
Epoch 200/500, Loss: 65.4769
Epoch 210/500, Loss: 63.0683
Epoch 220/500, Loss: 60.8339
Epoch 230/500, Loss: 58.7605
Epoch 240/500, Loss: 56.8323
Epoch 250/500, Loss: 55.0206
Epoch 260/500, Loss: 53.2811
Epoch 270/500, Loss: 51.5315
Epoch 280/500, Loss: 49.6186
Epoch 290/500, Loss: 47.7224
Epoch 300/500, Loss: 45.6231
Epoch 310/500, Loss: 43.6113
Epoch 320/500, Loss: 41.7716
Epoch 330/500, Loss: 40.0531
Epoch 340/500, Loss: 38.4063
Epoch 350/500

In [17]:
# # --- 6. Previsão auto-regressiva (usando apenas o histórico)
# model.eval()
# history = list(serie[:train_points])
# predictions = []

# for _ in range(future_steps):
#     seq_input = torch.tensor(history[-seq_len:], dtype=torch.float32).unsqueeze(0).unsqueeze(-1).to(device)
#     with torch.no_grad():
#         next_val = model(seq_input).item()
#     predictions.append(next_val)
#     history.append(next_val)

# # --- 7. PLOT
# plt.figure(figsize=(15,5))
# plt.plot(t, serie, label="Série original", alpha=0.5)
# plt.plot(t[train_points:train_points+future_steps], predictions, label="Previsão MLP", color="r")
# plt.axvline(x=train_points, color='k', linestyle='--', label="Início da previsão")
# plt.legend()
# plt.title("Previsão de Série Temporal com MLP")
# plt.xlabel("Tempo (observações de 30min)")
# plt.ylabel("Valor")
# plt.grid(True)
# plt.show()

## Forecasting in batch

In [18]:
nodes = [53,  365,  382,  666,  701, 1326, 1404, 1569, 1916, 2617]

In [19]:
stops = {53: '125960550',
         365: '230565994',
         382: '258781031',
         666: '43768720',
         701: '44072192',
         1326: '44783654',
         1404: '44783914',
         1569: '44784438',
         1916: '45833547',
         2617: '47568123'}

In [20]:
fpath_root = "/mnt/data/marcos/data/node_regression_bus/data_split/"
test_dataset = load_datasets(f'{fpath_root}test.pkl')
# x (dados de input)
test_dataset[0].x.shape

torch.Size([2871, 280])

In [21]:
model_name = "GRU-batch"

In [22]:
scores_error = {'node': [], 'batch': [], 'mae': [], 'mse': [], 'r2': [], 'mape': []}
targets = {}
cost, time = 0, 0
dfs = []


for node in nodes:
    dfs_pred = []    
    for time, snapshot in tqdm(enumerate(test_dataset)):
        snapshot.to('cuda')
        predictions = []
        targets[node] = []
        #
        # (alterar aqui o modelo)
        #
        y_true = snapshot.y.cpu().data.numpy()
        x_input = snapshot.x[node,:]
        x_input = x_input.view(1, 280, 1)
        for j in range(y_true.shape[1]):
            
            with torch.no_grad():
                pred = model(x_input).item()

            # Atualiza o input com o valor predito (lag)
            x_input = x_input.squeeze(0)
            x_input[:-1] = x_input[1:].clone()  # <-- clone aqui é essencial
            x_input[-1] = pred
            x_input = x_input.unsqueeze(0)  # mantém formato [1, features]

                               
            predictions.append(pred)

        y_pred = np.array(predictions)
        y_true = y_true[node,:]
        
        # nao alterar mais abaixo
        
        scores_error['node'].append(stops[node])
        scores_error['batch'].append(time)
        scores_error['mse'].append(mean_squared_error(y_true, y_pred))
        scores_error['mae'].append(mean_absolute_error(y_true, y_pred))
        scores_error['r2'].append(r2_score(y_true, y_pred))
        scores_error['mape'].append(mean_absolute_percentage_error(y_true, y_pred))
        
        # targets.append({'true': y_true,
        #                 'pred': y_pred})
        targets[node].append({"input":  snapshot.x[node,:].cpu().numpy(), 
                              'true': y_true,
                              'pred': y_pred,
                              'node': stops[node]
                             })
        
        

38it [00:03, 10.16it/s]
38it [00:03, 10.19it/s]
38it [00:03, 10.23it/s]
38it [00:03, 10.00it/s]
38it [00:03, 10.05it/s]
38it [00:03,  9.99it/s]
38it [00:03,  9.95it/s]
38it [00:03, 10.02it/s]
38it [00:03, 10.05it/s]
38it [00:03, 10.03it/s]


In [23]:
with open(f'results/{model_name}-targets.pkl', 'wb') as f:
    pickle.dump(targets, f)

In [24]:
df_results = pd.DataFrame(scores_error)
df_results["model"] = model_name
df_results.to_parquet(f"results/{model_name}-batch.parquet", index=False)

In [25]:
plt.plot(targets[0]["true"], '--', label="time series")
plt.plot(targets[0]["pred"], '--', label="forecasting")
plt.legend()
plt.show()

KeyError: 0

In [ ]:
df_results = pd.DataFrame(scores_error)
df_results

In [ ]:
df_results.pivot_table(index=["node"], 
                       values=["mae", "mse", "r2", "mape"], 
                       aggfunc="mean").reset_index()

In [ ]:
df_results["mae"].mean(), df_results["mape"].mean(), df_results["mse"].mean(), df_results["r2"].mean()